# Polars II

In this notebook, we will work with the following:

- Aggregating data with `group_by()`.
- Creating firm-year summaries from article-level data.
- Adding identifiers.
- Joining multiple datasets.
- Checking whether joins behaved the way we expected.
- Exporting an analysis-ready table.


In [ ]:
# imports
from pathlib import Path

import pandas as pd
import polars as pl

In [ ]:
# path
DATA_DIR = Path("..") / "data"

## Starting point

In the prior notebook, we read a small Stata dataset, converted it to Polars, and cleaned it up. We'll recreate that same starting point here so this notebook can run on its own.

Depending on the data, we might prefer to gather the data and write it out using another notebook, or we may want to have it integrated into one notebook. Often, that choice is about how long it takes to get the data, or if it comes from a source that we download manually.

My shorthand is this:

- **Manual, heavy network use (e.g., web scraping), or API (often metered and expensive).** Gather it separately, and read in the data when assembling research-ready datasets. We run things a lot, and there's no gain in reworking things that aren't changing. I often run these as separate projects maintained apart from research projects as a kind of infrastructure.
- **Fast network use.** This is something like pulling data from WRDS. Chances are, I still gather it separately, and have my data assembly rely on the file. If it's truly small, fast, unmetered (or not expensive), and/or requiring timeliness, I may have the data retrieved on use.
- **Computed for this project.** I tend to like to work with the original data and then do all of the processing in one place. When I don't do that, like the infrastructure data projects, I try to keep the data as close to the original form as I can in the infrastructure project and then transform in the research project.

However you do it, capture anything that changes from the original data in code. That's hard evidence of what you did, and it preserves your ability to change processing.

In [ ]:
firmyear_pd = pd.read_stata(DATA_DIR / "firmyear.dta")

firmyear = (
    pl.from_pandas(firmyear_pd)
    .with_columns(
        pl.col("year").cast(pl.Int64),
        pl.col("count_of_employees").cast(pl.Int64),
    )
    .rename({"count_of_employees": "size_emp"})
    .sort("name", "year")
)

firmyear.head()

## Aggregating data

When working with data, we often need to summarize many records into the level of analysis for the project. This can be, for example, individual analyst estimates, segmented financial data, news articles, social media, and many more.

Here, we have article-level data from the New York Times and a firm-year dataset. The goal is to create firm-year article summaries, then join those summaries onto our firm-year data.

The [Polars aggregation documentation](https://docs.pola.rs/user-guide/expressions/aggregation/) has many more examples of `group_by()` and `agg()`.

In [ ]:
articles = pl.read_csv(DATA_DIR / "msft_nyt.csv", try_parse_dates=True)
articles.head()

In [ ]:
articles.select("id_ticker", "pub_date", "word_count", "headline.main")

The article dates are more granular than we need for a firm-year merge. We'll extract the year from the publication date.

In [ ]:
articles = articles.with_columns(
    pl.col("pub_date").dt.year().alias("year"),
)

articles.select("id_ticker", "year", "word_count", "headline.main")

Now we can summarize the article data by ticker and year. This changes the level of the data from one row per article to one row per ticker-year.

In [ ]:
article_summary = (
    articles.group_by("id_ticker", "year")
    .agg(
        pl.len().alias("article_count"),
        pl.col("word_count").mean().round(1).alias("wc_mean"),
        pl.col("word_count").sum().alias("wc_sum"),
    )
    .sort("id_ticker", "year")
)

article_summary.head()

## Adding identifiers

The firm-year data use firm names, while the stock and article data use tickers. As humans, we can make the inference that particular pairs are related, but we need to be explicit to Python about how to do it.

We'll make a tiny lookup table and join it onto the firm-year data.

In [ ]:
lookup = pl.DataFrame(
    {
        "name": ["Google", "Microsoft"],
        "id_ticker": ["goog", "msft"],
    }
)

lookup.head()

The [Polars join documentation](https://docs.pola.rs/user-guide/transformations/joins/) covers the main join types. Here, we want a left join because the firm-year data are our base dataset. That's common. If we start with base data at our intended level of analysis, we often do a lot of left joins, unless we're checking why some things are not merging.

The `validate="m:1"` argument states our expectation that many firm-year rows may match one lookup row.

In [ ]:
firmyear = firmyear.join(
    lookup,
    on="name",
    how="left",
    validate="m:1",
)

firmyear.head()

## Joining stock data

Next, we'll read the stock data and join beginning-of-year stock prices onto the firm-year data.

The column names do not match, but that's not a problem here. We can specify which columns to use from the left and right dataframes when we join. In Stata, this kind of mismatch often means opening a file, renaming variables, saving it, and then merging. This is way better.

In [ ]:
stock = pl.read_csv(DATA_DIR / "stock.csv")

stock.head()

In [ ]:
firmyear_stock = firmyear.join(
    stock,
    left_on=["id_ticker", "year"],
    right_on=["tic", "yr"],
    how="left",
    validate="1:1",
)

firmyear_stock.head()

## Checking joins

Joining data is one of those places where code can run successfully and still produce the wrong dataset. At the same time, it's quite common for data not to completely match in our field. So, after a join, we usually want to check what happened and decide whether it's fine or something to fix.

A few useful checks are:

1. Did the row count change unexpectedly?
1. Are the keys unique at the level we thought they were?
1. Which rows did not match?

In [ ]:
print(f"Before join: {firmyear.height} rows")
print(f"After join:  {firmyear_stock.height} rows")

In [ ]:
firmyear.group_by("id_ticker", "year").len().filter(pl.col("len") > 1)

We expect some missing stock prices here, because I only gathered Microsoft for this example.

That kind of missingness is not automatically a problem, but we need to know about it.

In [ ]:
firmyear_stock.filter(pl.col("price").is_null()).select(
    "name",
    "year",
    "id_ticker",
    "price",
)

Polars also has an anti join, which returns rows from the left table that do not match the right table.
That makes it a useful tool for diagnosing joins.

In [ ]:
firmyear.join(
    stock,
    left_on=["id_ticker", "year"],
    right_on=["tic", "yr"],
    how="anti",
).head()

## Joining article summaries

Now we'll join the article summaries onto the firm-year data.
Again, the key is `id_ticker` and `year`.

Because I only gathered Microsoft articles from 2018, most rows will not match.

In [ ]:
firmyear_full = firmyear_stock.join(
    article_summary,
    on=["id_ticker", "year"],
    how="left",
    validate="1:1",
)

firmyear_full.head(6)

For counts and sums, it can make sense to replace missing values with zero after a join. Before doing this, we want to be sure missing means a true zero, not a retrieval problem or a merge problem.

Here, we'll assume that missing is a true zero.

In [ ]:
firmyear_full = firmyear_full.with_columns(
    pl.col("article_count", "wc_sum").fill_null(0),
)

firmyear_full.head()

## Saving and exporting

Finally, we'll save the joined firm-year dataset.

The date in the filename is a simple form of versioning. Git is better for tracking code, but dated data outputs are often easier to communicate with coauthors who are not working from Git hashes (just about all of them!). Date is granular enough, because we aren't usually producing multiple versions of the same dataset in a day for distribution to coauthors.

In [ ]:
firmyear_full.write_csv(DATA_DIR / "2026-06-02_firmyear_analysis_polars.csv")
firmyear_full.write_parquet(DATA_DIR / "2026-06-02_firmyear_analysis_polars.parquet")

In [ ]:
pl.read_parquet(DATA_DIR / "2026-06-02_firmyear_analysis_polars.parquet").head()

## On your own

Try the same basic workflow on the provided data or on data of your own.

1. Create one grouped summary with `group_by()` and `agg()`.
1. Join that summary onto another dataframe.
1. Check the row count before and after the join.
1. Find rows that did not match.
1. Write the result with a dated filename.

In [ ]:
# 1-1 code

In [ ]:
# 1-2 code

In [ ]:
# 1-3 code